In [ ]:
# 02 — Data Preparation

## Machine Learning-Based Forecasting of Water Stress in Eswatini

This notebook prepares the hydroclimatic data used for the water-stress forecasting experiment.

The workflow includes:

1. Loading the processed daily hydroclimatic dataset.
2. Checking temporal coverage, missing values, and duplicate observations.
3. Converting the date variable to a consistent datetime format.
4. Aggregating daily environmental variables to monthly resolution.
5. Performing basic quality-control checks.
6. Saving the monthly dataset for target construction and feature engineering.

Water-stress target construction is intentionally performed in the next notebook to keep data preparation separate from model-target development.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

cwd = Path.cwd()

# Works whether the notebook kernel starts from the project
# root or from the notebooks directory.
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project data directory."
    )

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

DAILY_FILE = (
    PROCESSED_DIR /
    "mnjoli_water_stress_features_daily.csv"
)

MONTHLY_FILE = (
    PROCESSED_DIR /
    "mnjoli_water_stress_monthly.csv"
)

print("Current directory :", cwd)
print("Project root      :", PROJECT_ROOT)
print("Daily dataset     :", DAILY_FILE)
print("Monthly output    :", MONTHLY_FILE)
print("Daily file exists :", DAILY_FILE.exists())

Current directory : c:\Users\Para\Desktop\Eswatini_Water_Stress_ML\notebooks
Project root      : c:\Users\Para\Desktop\Eswatini_Water_Stress_ML
Daily dataset     : c:\Users\Para\Desktop\Eswatini_Water_Stress_ML\data\processed\mnjoli_water_stress_features_daily.csv
Monthly output    : c:\Users\Para\Desktop\Eswatini_Water_Stress_ML\data\processed\mnjoli_water_stress_monthly.csv
Daily file exists : True


In [8]:
df_daily = pd.read_csv(DAILY_FILE)

print("=" * 70)
print("RAW DAILY DATASET")
print("=" * 70)

print("Shape:", df_daily.shape)

print("\nColumns:")
for i, col in enumerate(df_daily.columns, 1):
    print(f"{i:02d}. {col}")

display(df_daily.head())

RAW DAILY DATASET
Shape: (4017, 26)

Columns:
01. date
02. precipitation_mm
03. dewpoint_temperature_2m
04. potential_evaporation_sum
05. runoff_sum
06. surface_pressure
07. surface_runoff_sum
08. surface_solar_radiation_downwards_sum
09. temperature_2m
10. temperature_2m_max
11. temperature_2m_min
12. u_component_of_wind_10m
13. v_component_of_wind_10m
14. volumetric_soil_water_layer_1
15. volumetric_soil_water_layer_2
16. pet_m
17. pet_mm
18. year
19. month
20. day_of_year
21. temperature_c
22. temperature_min_c
23. temperature_max_c
24. dewpoint_c
25. wind_speed
26. surface_pressure_kpa


,date,precipitation_mm,dewpoint_temperature_2m,potential_evaporation_sum,runoff_sum,surface_pressure,surface_runoff_sum,surface_solar_radiation_downwards_sum,temperature_2m,temperature_2m_max,...,pet_mm,year,month,day_of_year,temperature_c,temperature_min_c,temperature_max_c,dewpoint_c,wind_speed,surface_pressure_kpa
0,2015-01-01,0.0,291.253237,-0.005317,1.095235e-05,96857.947005,1.056180e-05,1.638664e+07,295.476449,299.686881,...,5.316992,2015,1,1,22.326449,19.471557,26.536881,18.103237,1.803615,96.857947
1,2015-01-02,0.0,292.773722,-0.006181,1.488502e-06,96789.007270,1.092895e-06,1.877256e+07,296.835059,301.037762,...,6.180669,2015,1,2,23.685059,19.882143,27.887762,19.623722,2.106165,96.789007
2,2015-01-03,0.0,293.917622,-0.006040,1.264124e-06,96780.185422,8.603941e-07,1.943779e+07,297.639751,302.467197,...,6.040124,2015,1,3,24.489751,20.370614,29.317197,20.767622,1.825774,96.780185
3,2015-01-04,0.0,293.504903,-0.007706,7.396248e-07,96754.536558,3.278793e-07,2.593199e+07,298.456794,303.712293,...,7.706183,2015,1,4,25.306794,21.499785,30.562293,20.354903,1.400170,96.754537
4,2015-01-05,0.0,292.642278,-0.009745,4.335323e-07,96622.785307,3.502042e-08,2.949817e+07,299.187555,305.122049,...,9.745081,2015,1,5,26.037555,21.028642,31.972049,19.492278,1.727800,96.622785


In [9]:
# ------------------------------------------------------------
# IDENTIFY DATE COLUMN
# ------------------------------------------------------------

possible_date_columns = [
    col for col in df_daily.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("Possible date columns:")
print(possible_date_columns)

Possible date columns:
['date']


In [10]:
DATE_COLUMN = "date"

df_daily[DATE_COLUMN] = pd.to_datetime(
    df_daily[DATE_COLUMN],
    errors="coerce"
)

df_daily = (
    df_daily
    .sort_values(DATE_COLUMN)
    .reset_index(drop=True)
)

print("\nDate range:")
print(df_daily[DATE_COLUMN].min())
print("to")
print(df_daily[DATE_COLUMN].max())

print(
    "\nInvalid dates:",
    df_daily[DATE_COLUMN].isna().sum()
)


Date range:
2015-01-01 00:00:00
to
2025-12-30 00:00:00

Invalid dates: 0


In [11]:
print("=" * 70)
print("DAILY DATA QUALITY CHECK")
print("=" * 70)

print("\nShape:")
print(df_daily.shape)

print("\nDuplicate dates:")
print(
    df_daily[DATE_COLUMN]
    .duplicated()
    .sum()
)

print("\nMissing values:")
missing_values = (
    df_daily
    .isna()
    .sum()
)

missing_values = (
    missing_values[
        missing_values > 0
    ]
    .sort_values(ascending=False)
)

if len(missing_values) == 0:
    print("NONE")
else:
    print(missing_values)


print("\nData types:")
print(df_daily.dtypes)


# ------------------------------------------------------------
# TEMPORAL CONTINUITY
# ------------------------------------------------------------

valid_dates = (
    df_daily[DATE_COLUMN]
    .dropna()
)

expected_dates = pd.date_range(
    start=valid_dates.min(),
    end=valid_dates.max(),
    freq="D"
)

observed_dates = pd.DatetimeIndex(
    valid_dates.unique()
)

missing_dates = (
    expected_dates
    .difference(observed_dates)
)

print("\n" + "=" * 70)
print("TEMPORAL COVERAGE")
print("=" * 70)

print(
    "Expected calendar days:",
    len(expected_dates)
)

print(
    "Unique observed dates:",
    len(observed_dates)
)

print(
    "Missing calendar days:",
    len(missing_dates)
)

if len(missing_dates) > 0:
    print("\nFirst 20 missing dates:")
    print(missing_dates[:20])
else:
    print(
        "\nNo missing calendar days detected."
    )

DAILY DATA QUALITY CHECK

Shape:
(4017, 26)

Duplicate dates:
0

Missing values:
NONE

Data types:
date                                     datetime64[us]
precipitation_mm                                float64
dewpoint_temperature_2m                         float64
potential_evaporation_sum                       float64
runoff_sum                                      float64
surface_pressure                                float64
surface_runoff_sum                              float64
surface_solar_radiation_downwards_sum           float64
temperature_2m                                  float64
temperature_2m_max                              float64
temperature_2m_min                              float64
u_component_of_wind_10m                         float64
v_component_of_wind_10m                         float64
volumetric_soil_water_layer_1                   float64
volumetric_soil_water_layer_2                   float64
pet_m                                           float64
pet_m

In [12]:
# ============================================================
# EXISTING MONTHLY DATASET — REFERENCE ONLY
# ============================================================

df_monthly_reference = pd.read_csv(
    MONTHLY_FILE,
    parse_dates=["month_date"]
)

df_monthly_reference = (
    df_monthly_reference
    .sort_values("month_date")
    .reset_index(drop=True)
)

print("=" * 70)
print("EXISTING MONTHLY REFERENCE DATASET")
print("=" * 70)

print("Shape:", df_monthly_reference.shape)

print(
    "Date range:",
    df_monthly_reference["month_date"].min(),
    "to",
    df_monthly_reference["month_date"].max()
)

print("\nColumns:")

for i, col in enumerate(
    df_monthly_reference.columns,
    1
):
    print(f"{i:02d}. {col}")

display(
    df_monthly_reference.head()
)

EXISTING MONTHLY REFERENCE DATASET
Shape: (132, 14)
Date range: 2015-01-01 00:00:00 to 2025-12-01 00:00:00

Columns:
01. month_date
02. precipitation_mm
03. pet_mm
04. temperature_mean_c
05. temperature_min_c
06. temperature_max_c
07. dewpoint_c
08. soil_moisture_layer1
09. soil_moisture_layer2
10. runoff_mm
11. surface_runoff_mm
12. solar_radiation
13. wind_speed
14. surface_pressure_kpa


,month_date,precipitation_mm,pet_mm,temperature_mean_c,temperature_min_c,temperature_max_c,dewpoint_c,soil_moisture_layer1,soil_moisture_layer2,runoff_mm,surface_runoff_mm,solar_radiation,wind_speed,surface_pressure_kpa
0,2015-01-01,108.399174,223.820343,24.244754,20.536907,29.074656,19.159322,0.254893,0.247987,0.000448,0.000436,2.080123e+07,1.792599,97.009791
1,2015-02-01,153.429877,205.196632,24.368461,19.930770,29.984754,18.570688,0.224155,0.216465,0.001506,0.001496,1.991791e+07,1.525280,96.886179
2,2015-03-01,66.667655,230.976277,24.528914,19.785820,30.321264,16.856486,0.150096,0.165199,0.000093,0.000083,1.899828e+07,1.499579,97.252747
3,2015-04-01,25.722274,158.938783,21.506630,17.276988,26.593867,15.441193,0.193402,0.175428,0.001043,0.001034,1.379342e+07,1.319609,97.352107
4,2015-05-01,6.401154,156.063646,20.217188,14.224896,26.989467,13.045341,0.155295,0.184315,0.000055,0.000046,1.443251e+07,1.215148,97.510695


In [13]:
# ============================================================
# RECONSTRUCT MONTHLY HYDROCLIMATIC DATA
# ============================================================

df_daily["month_date"] = (
    df_daily[DATE_COLUMN]
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_monthly_candidate = (
    df_daily
    .groupby(
        "month_date",
        as_index=False
    )
    .agg(
        precipitation_mm=(
            "precipitation_mm",
            "sum"
        ),

        pet_mm=(
            "pet_mm",
            "sum"
        ),

        temperature_mean_c=(
            "temperature_c",
            "mean"
        ),

        temperature_min_c=(
            "temperature_min_c",
            "mean"
        ),

        temperature_max_c=(
            "temperature_max_c",
            "mean"
        ),

        dewpoint_c=(
            "dewpoint_c",
            "mean"
        ),

        soil_moisture_layer1=(
            "volumetric_soil_water_layer_1",
            "mean"
        ),

        soil_moisture_layer2=(
            "volumetric_soil_water_layer_2",
            "mean"
        ),

        runoff_mm=(
            "runoff_sum",
            "sum"
        ),

        surface_runoff_mm=(
            "surface_runoff_sum",
            "sum"
        ),

        solar_radiation=(
            "surface_solar_radiation_downwards_sum",
            "mean"
        ),

        wind_speed=(
            "wind_speed",
            "mean"
        ),

        surface_pressure_kpa=(
            "surface_pressure_kpa",
            "mean"
        )
    )
)

print("=" * 70)
print("RECONSTRUCTED MONTHLY DATASET")
print("=" * 70)

print(
    "Shape:",
    df_monthly_candidate.shape
)

print(
    "Date range:",
    df_monthly_candidate["month_date"].min(),
    "to",
    df_monthly_candidate["month_date"].max()
)

display(
    df_monthly_candidate.head()
)

RECONSTRUCTED MONTHLY DATASET
Shape: (132, 14)
Date range: 2015-01-01 00:00:00 to 2025-12-01 00:00:00


,month_date,precipitation_mm,pet_mm,temperature_mean_c,temperature_min_c,temperature_max_c,dewpoint_c,soil_moisture_layer1,soil_moisture_layer2,runoff_mm,surface_runoff_mm,solar_radiation,wind_speed,surface_pressure_kpa
0,2015-01-01,108.399174,223.820343,24.244754,20.536907,29.074656,19.159322,0.254893,0.247987,0.000448,0.000436,2.080123e+07,1.792599,97.009791
1,2015-02-01,153.429877,205.196632,24.368461,19.930770,29.984754,18.570688,0.224155,0.216465,0.001506,0.001496,1.991791e+07,1.525280,96.886179
2,2015-03-01,66.667655,230.976277,24.528914,19.785820,30.321264,16.856486,0.150096,0.165199,0.000093,0.000083,1.899828e+07,1.499579,97.252747
3,2015-04-01,25.722274,158.938783,21.506630,17.276988,26.593867,15.441193,0.193402,0.175428,0.001043,0.001034,1.379342e+07,1.319609,97.352107
4,2015-05-01,6.401154,156.063646,20.217188,14.224896,26.989467,13.045341,0.155295,0.184315,0.000055,0.000046,1.443251e+07,1.215148,97.510695


In [14]:
# ============================================================
# VERIFY AGAINST EXISTING MONTHLY DATASET
# ============================================================

common_columns = [
    col
    for col in df_monthly_candidate.columns
    if col in df_monthly_reference.columns
    and col != "month_date"
]

comparison = (
    df_monthly_candidate[
        ["month_date"] + common_columns
    ]
    .merge(
        df_monthly_reference[
            ["month_date"] + common_columns
        ],
        on="month_date",
        suffixes=("_new", "_old"),
        how="inner"
    )
)

comparison_results = []

for column in common_columns:

    new_values = comparison[
        f"{column}_new"
    ]

    old_values = comparison[
        f"{column}_old"
    ]

    absolute_difference = (
        new_values - old_values
    ).abs()

    comparison_results.append({
        "Variable": column,
        "Max_Abs_Difference":
            absolute_difference.max(),
        "Mean_Abs_Difference":
            absolute_difference.mean(),
        "Matches":
            np.allclose(
                new_values,
                old_values,
                rtol=1e-6,
                atol=1e-8,
                equal_nan=True
            )
    })

comparison_results = pd.DataFrame(
    comparison_results
)

print("=" * 70)
print("MONTHLY REPRODUCIBILITY CHECK")
print("=" * 70)

print(
    comparison_results
    .to_string(index=False)
)

MONTHLY REPRODUCIBILITY CHECK
            Variable  Max_Abs_Difference  Mean_Abs_Difference  Matches
    precipitation_mm        5.684342e-14         1.549266e-15     True
              pet_mm        5.684342e-14         7.105427e-15     True
  temperature_mean_c        3.552714e-15         5.652044e-16     True
   temperature_min_c        3.552714e-15         4.171747e-16     True
   temperature_max_c        3.552714e-15         9.420074e-16     True
          dewpoint_c        3.552714e-15         4.575465e-16     True
soil_moisture_layer1        9.714451e-17         3.984607e-17     True
soil_moisture_layer2        8.326673e-17         3.742797e-17     True
           runoff_mm        9.822872e-17         3.412712e-17     True
   surface_runoff_mm        9.887924e-17         3.185917e-17     True
     solar_radiation        3.725290e-09         6.349927e-10     True
          wind_speed        4.440892e-16         6.223978e-17     True
surface_pressure_kpa        1.421085e-14       

In [15]:
# ============================================================
# MONTHLY DATA QUALITY CHECK
# ============================================================

print("=" * 70)
print("MONTHLY DATA QUALITY CHECK")
print("=" * 70)

print("Observations:", len(df_monthly_candidate))
print("Variables:", len(df_monthly_candidate.columns))

print(
    "Period:",
    df_monthly_candidate["month_date"].min(),
    "to",
    df_monthly_candidate["month_date"].max()
)

print(
    "Duplicate months:",
    df_monthly_candidate["month_date"]
    .duplicated()
    .sum()
)

print(
    "Missing values:",
    df_monthly_candidate
    .isna()
    .sum()
    .sum()
)

# Expected monthly sequence
expected_months = pd.date_range(
    start=df_monthly_candidate["month_date"].min(),
    end=df_monthly_candidate["month_date"].max(),
    freq="MS"
)

observed_months = pd.DatetimeIndex(
    df_monthly_candidate["month_date"]
)

missing_months = expected_months.difference(
    observed_months
)

print(
    "Expected months:",
    len(expected_months)
)

print(
    "Observed months:",
    len(observed_months)
)

print(
    "Missing months:",
    len(missing_months)
)

if len(missing_months) > 0:
    print("\nMissing monthly observations:")
    print(missing_months)
else:
    print("\nNo missing monthly observations detected.")

MONTHLY DATA QUALITY CHECK
Observations: 132
Variables: 14
Period: 2015-01-01 00:00:00 to 2025-12-01 00:00:00
Duplicate months: 0
Missing values: 0
Expected months: 132
Observed months: 132
Missing months: 0

No missing monthly observations detected.


In [16]:
# ============================================================
# MONTHLY DESCRIPTIVE STATISTICS
# ============================================================

print("=" * 70)
print("MONTHLY HYDROCLIMATIC SUMMARY")
print("=" * 70)

monthly_summary = (
    df_monthly_candidate
    .drop(columns="month_date")
    .describe()
    .T
)

display(
    monthly_summary[
        ["count", "mean", "std", "min", "max"]
    ].round(4)
)

MONTHLY HYDROCLIMATIC SUMMARY


,count,mean,std,min,max
precipitation_mm,132.0,6.563670e+01,7.175420e+01,1.905100e+00,4.932320e+02
pet_mm,132.0,1.895946e+02,4.102900e+01,1.024987e+02,2.958641e+02
temperature_mean_c,132.0,2.123710e+01,2.868900e+00,1.484690e+01,2.611460e+01
temperature_min_c,132.0,1.606610e+01,3.666400e+00,8.803500e+00,2.148820e+01
temperature_max_c,132.0,2.710420e+01,2.358500e+00,2.105540e+01,3.226970e+01
dewpoint_c,132.0,1.443380e+01,3.958200e+00,6.023200e+00,2.050770e+01
soil_moisture_layer1,132.0,1.935000e-01,6.060000e-02,1.074000e-01,3.724000e-01
soil_moisture_layer2,132.0,1.987000e-01,5.280000e-02,1.353000e-01,3.675000e-01
runoff_mm,132.0,2.100000e-03,5.600000e-03,0.000000e+00,3.240000e-02
surface_runoff_mm,132.0,2.100000e-03,5.400000e-03,0.000000e+00,2.990000e-02


In [17]:
# ============================================================
# SAVE VERIFIED MONTHLY DATASET
# ============================================================

df_monthly = (
    df_monthly_candidate
    .copy()
    .sort_values("month_date")
    .reset_index(drop=True)
)

df_monthly.to_csv(
    MONTHLY_FILE,
    index=False
)

print("=" * 70)
print("MONTHLY DATASET SAVED")
print("=" * 70)

print("File:")
print(MONTHLY_FILE)

print("\nShape:")
print(df_monthly.shape)

print("\nPeriod:")
print(
    df_monthly["month_date"].min(),
    "to",
    df_monthly["month_date"].max()
)

print(
    "\nFile exists:",
    MONTHLY_FILE.exists()
)

MONTHLY DATASET SAVED
File:
c:\Users\Para\Desktop\Eswatini_Water_Stress_ML\data\processed\mnjoli_water_stress_monthly.csv

Shape:
(132, 14)

Period:
2015-01-01 00:00:00 to 2025-12-01 00:00:00

File exists: True


In [18]:
# ============================================================
# FINAL FILE VERIFICATION
# ============================================================

df_check = pd.read_csv(
    MONTHLY_FILE,
    parse_dates=["month_date"]
)

print("=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

print("Saved shape:", df_check.shape)

print(
    "Period:",
    df_check["month_date"].min(),
    "to",
    df_check["month_date"].max()
)

print(
    "Missing values:",
    df_check.isna().sum().sum()
)

print(
    "Duplicate months:",
    df_check["month_date"]
    .duplicated()
    .sum()
)

print("\nFirst 5 observations:")
display(df_check.head())

print("\nLast 5 observations:")
display(df_check.tail())

FINAL VERIFICATION
Saved shape: (132, 14)
Period: 2015-01-01 00:00:00 to 2025-12-01 00:00:00
Missing values: 0
Duplicate months: 0

First 5 observations:


,month_date,precipitation_mm,pet_mm,temperature_mean_c,temperature_min_c,temperature_max_c,dewpoint_c,soil_moisture_layer1,soil_moisture_layer2,runoff_mm,surface_runoff_mm,solar_radiation,wind_speed,surface_pressure_kpa
0,2015-01-01,108.399174,223.820343,24.244754,20.536907,29.074656,19.159322,0.254893,0.247987,0.000448,0.000436,2.080123e+07,1.792599,97.009791
1,2015-02-01,153.429877,205.196632,24.368461,19.930770,29.984754,18.570688,0.224155,0.216465,0.001506,0.001496,1.991791e+07,1.525280,96.886179
2,2015-03-01,66.667655,230.976277,24.528914,19.785820,30.321264,16.856486,0.150096,0.165199,0.000093,0.000083,1.899828e+07,1.499579,97.252747
3,2015-04-01,25.722274,158.938783,21.506630,17.276988,26.593867,15.441193,0.193402,0.175428,0.001043,0.001034,1.379342e+07,1.319609,97.352107
4,2015-05-01,6.401154,156.063646,20.217188,14.224896,26.989467,13.045341,0.155295,0.184315,0.000055,0.000046,1.443251e+07,1.215148,97.510695



Last 5 observations:


,month_date,precipitation_mm,pet_mm,temperature_mean_c,temperature_min_c,temperature_max_c,dewpoint_c,soil_moisture_layer1,soil_moisture_layer2,runoff_mm,surface_runoff_mm,solar_radiation,wind_speed,surface_pressure_kpa
127,2025-08-01,8.244601,185.756645,19.124710,12.603221,26.331313,10.471165,0.114564,0.141975,0.000011,0.000008,1.632786e+07,1.550565,97.777980
128,2025-09-01,47.746935,201.017186,21.450627,16.138575,27.873384,13.667581,0.135774,0.138409,0.000130,0.000128,1.604192e+07,1.701459,97.663455
129,2025-10-01,31.915177,215.955591,21.624638,16.134544,27.788578,14.347702,0.174200,0.159271,0.000522,0.000520,1.820500e+07,1.701968,97.554029
130,2025-11-01,219.932661,185.266684,21.798029,17.576829,26.508866,16.654164,0.261434,0.247278,0.000612,0.000610,1.705413e+07,1.634452,97.419123
131,2025-12-01,142.493023,188.852572,22.934049,19.015264,27.406045,18.886109,0.343702,0.328864,0.018428,0.017767,1.908989e+07,1.628361,97.198083


In [19]:
## Data Preparation Summary

The processed daily hydroclimatic dataset contained 4,017 consecutive daily observations with no missing values, duplicate dates, or temporal gaps.

Daily observations were aggregated to monthly resolution according to the physical characteristics of each variable. Precipitation, potential evapotranspiration (PET), runoff, and surface runoff were accumulated using monthly sums, while temperature, dew point, soil moisture, solar radiation, wind speed, and surface pressure were represented using monthly means.

The reconstructed monthly variables were compared against the previously generated monthly dataset. All variables matched within numerical floating-point tolerance, confirming that the data-preparation workflow is reproducible.

The resulting monthly hydroclimatic dataset is used in the subsequent notebook for leakage-free water-stress target construction, temporal feature engineering, and preparation of the one-month-ahead forecasting dataset.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3842404795.py, line 3)